In [20]:
import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, average_precision_score, precision_score, recall_score

In [ ]:
# Cell 2: Load datasets and define key columns
train_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

outcome_col = "conversion"
treatment_col = "treatment"

feature_cols = [c for c in train_df.columns if c != outcome_col]
t_learner_feature_cols = [c for c in feature_cols if c != treatment_col]

X_train = train_df[t_learner_feature_cols].copy()
y_train = train_df[outcome_col].astype(int)
w_train = train_df[treatment_col].astype(int)
X_test = test_df[t_learner_feature_cols].copy()
y_test = test_df[outcome_col].astype(int)
w_test = test_df[treatment_col].astype(int)

print("Train rows:", len(train_df), "| Test rows:", len(test_df))

Train rows: 800000 | Test rows: 200000


In [22]:
# Cell 3: Train LightGBM T-learner without encoding categoricals
numeric_cols = X_train.select_dtypes(include=[np.number, "bool"]).columns.tolist()
categorical_cols = [c for c in X_train.columns if c not in numeric_cols]

# Use pandas categorical dtype so LightGBM can handle categorical features natively.
for col in categorical_cols:
    combined = pd.concat([X_train[col], X_test[col]], axis=0).astype("string")
    categories = pd.Index(combined.dropna().unique())
    X_train[col] = pd.Categorical(X_train[col].astype("string"), categories=categories)
    X_test[col] = pd.Categorical(X_test[col].astype("string"), categories=categories)

treated_mask = w_train == 1
control_mask = w_train == 0

t_learner_treated = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

t_learner_control = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.9,
    colsample_bytree=0.9,
    random_state=42,
    n_jobs=-1,
)

t_learner_treated.fit(X_train.loc[treated_mask], y_train.loc[treated_mask], categorical_feature=categorical_cols)
t_learner_control.fit(X_train.loc[control_mask], y_train.loc[control_mask], categorical_feature=categorical_cols)
print("LightGBM T-learner trained (separate treated/control models, native categorical handling).")

[LightGBM] [Info] Number of positive: 1326, number of negative: 23359
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.002614 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1935
[LightGBM] [Info] Number of data points in the train set: 24685, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.053717 -> initscore=-2.868815
[LightGBM] [Info] Start training from score -2.868815
[LightGBM] [Info] Number of positive: 1008, number of negative: 774307
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.022631 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1672
[LightGBM] [Info] Number of data points in the train set: 775315, number of used features: 14
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.001300 -> initscore=-6.644000
[L

In [23]:
# Cell 4: Evaluate factual performance on test.csv
mu1_test = t_learner_treated.predict_proba(X_test)[:, 1]
mu0_test = t_learner_control.predict_proba(X_test)[:, 1]
factual_proba = np.where(w_test.values == 1, mu1_test, mu0_test)
factual_pred = (factual_proba >= 0.5).astype(int)

precision = precision_score(y_test, factual_pred, zero_division=0)
recall = recall_score(y_test, factual_pred, zero_division=0)
accuracy = accuracy_score(y_test, factual_pred)
auprc = average_precision_score(y_test, factual_proba)

print("Test metrics on test.csv")
print(f"Precision: {precision:.4f}")
print(f"Recall:    {recall:.4f}")
print(f"Accuracy:  {accuracy:.4f}")
print(f"AUPRC:     {auprc:.4f}")

Test metrics on test.csv
Precision: 0.4790
Recall:    0.1955
Accuracy:  0.9970
AUPRC:     0.2741


In [24]:
# Cell 5: Counterfactual conversion predictions from T-learner
mu1 = t_learner_treated.predict_proba(X_test)[:, 1]  # P(conversion | do(treatment=1), x)
mu0 = t_learner_control.predict_proba(X_test)[:, 1]  # P(conversion | do(treatment=0), x)
uplift = mu1 - mu0

results = test_df.copy()
results["p_conversion_treated"] = mu1
results["p_conversion_control"] = mu0
results["uplift"] = uplift
results

,f0,f1,f2,f3,f4,f5,f6,f7,f8,f9,f10,f11,treatment,conversion,visit,exposure,p_conversion_treated,p_conversion_control,uplift
0,23.032518,10.059654,8.214383,4.679882,10.280525,4.115453,-12.422866,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0,0.000042,4.355040e-07,0.000041
1,25.600311,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,1,0,0,0,0.000046,4.382485e-07,0.000046
2,19.523195,10.059654,8.639891,3.359763,10.280525,4.115453,-7.301017,4.833815,3.878372,25.240993,5.300375,-0.168679,1,0,0,0,0.000041,6.846597e-07,0.000041
3,12.616365,10.059654,8.422836,4.679882,10.280525,4.115453,0.294443,4.833815,3.835851,18.380112,5.300375,-0.168679,1,0,0,0,0.000046,5.757495e-07,0.000046
4,20.454730,10.059654,8.847605,3.907662,10.280525,4.115453,-10.006574,4.833815,3.899112,13.190056,5.300375,-0.168679,1,0,0,0,0.000029,5.699613e-07,0.000028
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,12.616365,10.059654,9.040947,4.679882,10.280525,4.115453,0.294443,4.833815,3.943716,13.190056,5.300375,-0.168679,1,0,0,0,0.000024,4.219120e-07,0.000024
199996,24.811454,10.059654,8.214383,4.679882,10.280525,4.115453,-1.288207,4.833815,3.971858,13.190056,5.300375,-0.168679,0,0,0,0,0.000046,3.789734e-07,0.000046
199997,22.928756,10.059654,8.378748,4.679882,10.280525,3.013064,-15.877431,11.560131,3.771810,46.672202,5.300375,-0.168679,0,0,0,0,0.000019,6.394987e-07,0.000018
199998,12.616365,10.059654,8.841462,4.679882,10.280525,4.115453,0.294443,4.833815,3.943716,13.190056,5.300375,-0.168679,1,0,0,0,0.000042,6.368677e-07,0.000042


In [26]:
# Cell 6: Save trained T-learner models to PKL
import pickle

t_learner_bundle = {
    "treated_model": t_learner_treated,
    "control_model": t_learner_control,
    "feature_cols": t_learner_feature_cols,
    "categorical_cols": categorical_cols,
    "treatment_col": treatment_col,
    "outcome_col": outcome_col,
}

with open("t_learner.pkl", "wb") as f:
    pickle.dump(t_learner_bundle, f)

print("Saved t_learner.pkl")

Saved t_learner.pkl
